# Day 19 — Intro to Structured Streaming

**What you will learn:**
- The continuous DataFrame model: a stream as an unbounded table
- Reading from streaming sources: file, socket, Kafka, rate
- Output modes: `append`, `complete`, `update`
- Triggers: `processingTime`, `once`, `availableNow`, `continuous`
- Writing to sinks: console, file, Kafka, Delta
- `checkpointLocation` — why it is mandatory

**Exam relevance:** Streaming (10%) — output modes, triggers, and checkpoint semantics are the most tested streaming topics.

In [ ]:
from pyspark.sql.functions import col, expr, window, count, sum, avg
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import tempfile, os, time

CHECKPOINT = tempfile.mkdtemp(prefix="stream_ckpt_")
OUTPUT_DIR = tempfile.mkdtemp(prefix="stream_out_")
SOURCE_DIR = tempfile.mkdtemp(prefix="stream_src_")

print(f"Checkpoint: {CHECKPOINT}")
print(f"Output:     {OUTPUT_DIR}")

## 1. The Unbounded Table Mental Model

Structured Streaming models a stream as an **ever-growing table**.

```
New data arrives → appended as rows to the input table
                 → Spark re-runs the query on new rows
                 → results written to output table (sink)
```

The same DataFrame API works for batch AND streaming — you just change the source and add `.writeStream`.

## 2. Output Modes

| Mode | What is written each trigger | Supported when |
|---|---|---|
| `append` | Only new rows added since last trigger | Queries without aggregation, or with watermark |
| `complete` | **Entire** result table every trigger | Only with aggregation |
| `update` | Only rows that changed since last trigger | With or without aggregation |

> **Exam tip:** `complete` requires aggregation — it rewrites the full result table. `append` is the default and most common.

## 3. Reading from a File Source

In [ ]:
# Declare schema — required for streaming file sources (no schema inference)
order_schema = StructType([
    StructField("order_id",   StringType(),  True),
    StructField("customer",   StringType(),  True),
    StructField("amount",     IntegerType(), True),
    StructField("region",     StringType(),  True),
    StructField("event_time", TimestampType(), True),
])

# Write a seed file so the stream has data to process
seed_data = spark.createDataFrame([
    ("ORD-001", "Alice", 150, "EU"),
    ("ORD-002", "Bob",    89, "US"),
    ("ORD-003", "Carol", 220, "EU"),
], ["order_id", "customer", "amount", "region"])

from pyspark.sql.functions import current_timestamp
seed_data.withColumn("event_time", current_timestamp()) \
    .write.mode("overwrite").json(SOURCE_DIR)

print(f"Seed data written to {SOURCE_DIR}")

In [ ]:
# Define the streaming read
stream_df = spark.readStream \
    .schema(order_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(SOURCE_DIR)

print("Is streaming:", stream_df.isStreaming)
stream_df.printSchema()

## 4. Triggers

Triggers control **when** the micro-batch runs:

| Trigger | Behaviour |
|---|---|
| `processingTime="5 seconds"` | Run every 5 seconds (default: as fast as possible) |
| `once=True` | Run one micro-batch then stop (legacy) |
| `availableNow=True` | Process all available data, stop (Spark 3.3+, replaces `once`) |
| `continuous="1 second"` | Experimental: low-latency continuous processing |

## 5. Writing to Console Sink (Development Only)

In [ ]:
query = stream_df \
    .filter(col("amount") > 100) \
    .writeStream \
    .format("console") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CHECKPOINT, "console")) \
    .start()

query.awaitTermination(timeout=30)
print("Console query done.")

## 6. Writing to File Sink

In [ ]:
query2 = stream_df \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("path", OUTPUT_DIR) \
    .option("checkpointLocation", os.path.join(CHECKPOINT, "parquet")) \
    .start()

query2.awaitTermination(timeout=30)

print("Output rows:", spark.read.parquet(OUTPUT_DIR).count())
spark.read.parquet(OUTPUT_DIR).show()

## 7. checkpointLocation — Why It Is Mandatory

The checkpoint directory stores:
1. **Offsets** — which data has already been read (prevents duplicate processing)
2. **State** — intermediate aggregation state for stateful operations
3. **Commit log** — confirms which micro-batches succeeded

Without a checkpoint:
- Restarting the stream re-processes all data from the beginning
- Stateful aggregations lose their accumulated state

> **Exam tip:** `checkpointLocation` is **required** for fault tolerance and exactly-once semantics. Always set it in production.

## 8. Reading from Rate Source (Testing Streams)

In [ ]:
# Rate source: generates (timestamp, value) at a fixed rate
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

rate_df.printSchema()

# Count rows per second — aggregation with complete mode
from pyspark.sql.functions import second, from_unixtime, unix_timestamp

rate_query = rate_df \
    .writeStream \
    .format("console") \
    .outputMode("update") \
    .trigger(processingTime="2 seconds") \
    .option("checkpointLocation", os.path.join(CHECKPOINT, "rate")) \
    .start()

# Let it run for a few seconds, then stop
time.sleep(6)
rate_query.stop()
print("Rate query stopped.")

## 9. StreamingQuery Status and Progress

In [ ]:
# After a query runs, inspect its last progress
if query2.lastProgress:
    import json
    print(json.dumps(query2.lastProgress, indent=2))
else:
    print("No progress data available (query may not have run).")

---

## Day 19 Checklist

- [ ] Know the unbounded table mental model
- [ ] Know all 3 output modes and when each is valid
- [ ] Used `availableNow=True` trigger to process and stop
- [ ] Set `checkpointLocation` — know what it stores and why it's required
- [ ] Wrote to console and file sinks
- [ ] Know that streaming requires explicit schema (no inference)
- [ ] Used `rate` source for testing

**Next:** Day 20 — Streaming Aggregations & Watermarks